# 41 - Silver labels retrained on the expanded, 101-query gold standard

Notebook 39 trained the silver-labelling classifier on 419 gold candidates from only 5 queries, and notebook 40's spot-check found this generalised poorly to the other 96 (65.9% accuracy, barely above the naive baseline), because the classifier had never seen a single example from those topics.

Notebook 40's active-learning queue has since added labelled candidates from all 101 queries, bringing the gold standard to 1,429 candidates covering every query. This notebook repeats notebook 39's approach (same classifier, same deployment-style fit on all available labels) but on the expanded gold standard, then repeats notebook 39's spot-check methodology on a **fresh** sample to see whether accuracy actually improves once training data spans every query topic instead of five.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

OUTPUT_DIR = Path("result/41_silver_labels_expanded_gold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2  # same strict "highly relevant" definition used in notebooks 38/39

gold = pd.read_json("result/40_active_learning_labeling_queue/expanded_gold_labels.json")
features = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")
feature_cols = ["score_minilm", "score_linq", "score_gte", "score_bm25",
                "invrank_minilm", "invrank_linq", "invrank_gte", "invrank_bm25",
                "reranker_score", "n_channels"]

labeled = gold.merge(features, on=["query_id", "domain"], how="inner")
labeled["relevant"] = (labeled["gold_label"] >= RELEVANT_THRESHOLD).astype(int)

print(f"Expanded gold candidates matched to features: {len(labeled)}/{len(gold)}")
print(f"Queries covered: {labeled['query_id'].nunique()}/101")
print(f"Relevant (gold_label >= {RELEVANT_THRESHOLD}): {labeled['relevant'].sum()}/{len(labeled)}")

Expanded gold candidates matched to features: 1427/1429
Queries covered: 101/101
Relevant (gold_label >= 2): 984/1427


In [2]:
# Deployment model: fit on ALL available gold labels, no held-out split (same reasoning as
# notebook 39 -- this model is meant to be applied at full scale, not evaluated in isolation).
X_train = labeled[feature_cols].values
y_train = labeled["relevant"].values

final_gbdt = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight="balanced", random_state=0)
final_gbdt.fit(X_train, y_train)

final_logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
final_logreg.fit(X_train, y_train)

print("Final model trained on all", len(labeled), "gold-labeled candidates spanning", labeled["query_id"].nunique(), "queries.")
print("Feature weights (logistic regression, for reference):")
for name, coef in sorted(zip(feature_cols, final_logreg.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"    {name:<16} {coef:+.3f}")

Final model trained on all 1427 gold-labeled candidates spanning 101 queries.
Feature weights (logistic regression, for reference):
    invrank_bm25     -2.246
    score_linq       +1.381
    invrank_minilm   -1.095
    score_minilm     -0.703
    score_gte        -0.701
    n_channels       +0.394
    invrank_linq     +0.172
    reranker_score   +0.146
    invrank_gte      +0.072
    score_bm25       -0.069


In [3]:
# Predict for every candidate across all 101 queries -- zero additional API cost.
X_all = features[feature_cols].values
features["silver_prob_relevant"] = final_gbdt.predict_proba(X_all)[:, 1]
features["silver_label"] = (features["silver_prob_relevant"] >= 0.5).astype(int)

gold_keys = set(zip(labeled["query_id"], labeled["domain"]))
features["tier"] = ["gold" if (q, d) in gold_keys else "silver" for q, d in zip(features["query_id"], features["domain"])]

silver_path = OUTPUT_DIR / "silver_labels_v2.json"
features.to_json(silver_path, orient="records", indent=2)

print(f"Total candidates: {len(features)}")
print(features["tier"].value_counts())
print(f"Saved -> {silver_path}")
print()
print("Silver-only predicted-relevant rate:")
print(features[features["tier"] == "silver"]["silver_label"].value_counts(normalize=True))

Total candidates: 173262
tier
silver    171835
gold        1427
Name: count, dtype: int64
Saved -> result/41_silver_labels_expanded_gold/silver_labels_v2.json

Silver-only predicted-relevant rate:
silver_label
1    0.612401
0    0.387599
Name: proportion, dtype: float64


## Fresh spot-check against the stronger judge pair

Same methodology as notebook 39's spot-check (`claude-sonnet-5` + `gpt-5.4`, unanimous agreement only), but a **new** random sample, since the old spot-check candidates may now be too close to (or overlap with) the expanded gold set. Also reuses the enriched prompt fields (state, district, nace_code, etc.) fixed in notebook 40.

In [4]:
import os, time
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

SPOT_CHECK_N = 150

corpus = pd.read_csv("dataset/company_corpus.csv")
# company_corpus.csv is one row per domain (see notebook 40) -- join company metadata on domain
# alone, and query text on query_id alone, not on the (query_id, domain) pair.
meta_cols = ["domain", "name", "country", "state", "district", "municipality",
             "organization_type", "organization_size", "summary", "summary_keywords", "nace_code"]
company_meta = corpus[meta_cols].drop_duplicates(subset="domain")
query_text = corpus[["query_id", "query"]].drop_duplicates(subset="query_id")

silver_only = features[features["tier"] == "silver"].copy()
silver_only = silver_only[silver_only["domain"].isin(company_meta["domain"])]
spot_sample = silver_only.sample(n=SPOT_CHECK_N, random_state=1).reset_index(drop=True)  # different seed than notebook 39 -- a fresh sample
spot_sample = spot_sample.merge(company_meta, on="domain", how="left")
spot_sample = spot_sample.merge(query_text, on="query_id", how="left")
print(f"Fresh spot-check sample: {len(spot_sample)} silver-labeled candidates")
print(f"Rows missing metadata: {spot_sample['name'].isna().sum()} (should be 0)")

Fresh spot-check sample: 150 silver-labeled candidates
Rows missing metadata: 0 (should be 0)


In [5]:
ENRICHED_JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Country: {country}
State/region: {state}
District: {district}
Municipality: {municipality}
Organization type: {organization_type}
Organization size: {organization_size}
NACE industry code: {nace_code}
Summary: {summary}
Summary keywords: {summary_keywords}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant (a strong, direct match for the query)
1 = partially relevant (related but not a strong direct match)
0 = not relevant

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def build_prompt(row):
    fields = {c: row.get(c, "") if pd.notna(row.get(c, "")) else "unknown" for c in
              ["query", "name", "country", "state", "district", "municipality",
               "organization_type", "organization_size", "nace_code", "summary", "summary_keywords"]}
    return ENRICHED_JUDGE_PROMPT_TEMPLATE.format(**fields)


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"


def judge_openai_gpt54(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-5.4", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude_sonnet5(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={
            "model": "claude-sonnet-5", "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=60,
    )
    resp.raise_for_status()
    # Sonnet 5 runs adaptive thinking by default, so content[0] may be a "thinking" block,
    # not "text" -- find the actual text block instead of assuming it's first.
    content_blocks = resp.json()["content"]
    text_block = next((b["text"] for b in content_blocks if b.get("type") == "text"), None)
    if text_block is None:
        return None, f"NO TEXT BLOCK: {content_blocks}"
    return parse_judge_reply(text_block)


spot_check_cache_path = OUTPUT_DIR / "spot_check_cache.json"
spot_results = json.load(open(spot_check_cache_path)) if spot_check_cache_path.exists() else {}

for i, row in spot_sample.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = spot_results.get(key, {})
    prompt = build_prompt(row)

    if "gpt54" not in entry:
        try:
            label, reason = judge_openai_gpt54(prompt)
            entry["gpt54"] = {"label": label, "reason": reason}
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [gpt-5.4] error on {row['domain']}: {e}")
    if "sonnet5" not in entry:
        try:
            label, reason = judge_claude_sonnet5(prompt)
            entry["sonnet5"] = {"label": label, "reason": reason}
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [sonnet5] error on {row['domain']}: {e}")

    spot_results[key] = entry
    json.dump(spot_results, open(spot_check_cache_path, "w"), indent=2, default=str)  # save after every row

    if (i + 1) % 25 == 0 or (i + 1) == len(spot_sample):
        print(f"  {i+1}/{len(spot_sample)} spot-checked")
    time.sleep(0.2)

print("Done.")

  25/150 spot-checked
  50/150 spot-checked
  75/150 spot-checked
  100/150 spot-checked
  125/150 spot-checked
  150/150 spot-checked
Done.


In [6]:
rows = []
for i, row in spot_sample.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = spot_results.get(key, {})
    if "gpt54" not in entry or "sonnet5" not in entry:
        continue
    gpt_label, sonnet_label = entry["gpt54"]["label"], entry["sonnet5"]["label"]
    if gpt_label is None or sonnet_label is None:
        continue
    # Unanimous agreement between the two strong judges counts as the "true" label;
    # disagreements are excluded, not guessed at (same principle as notebooks 35/39).
    if gpt_label != sonnet_label:
        continue
    true_relevant = int(gpt_label >= RELEVANT_THRESHOLD)
    rows.append({
        "query_id": row["query_id"], "domain": row["domain"],
        "silver_label": row["silver_label"], "true_relevant": true_relevant,
    })

spot_eval = pd.DataFrame(rows)
accuracy = (spot_eval["silver_label"] == spot_eval["true_relevant"]).mean() if len(spot_eval) else float("nan")
naive_baseline = spot_eval["true_relevant"].value_counts(normalize=True).max() if len(spot_eval) else float("nan")

print(f"Spot-checked with unanimous strong-judge agreement: {len(spot_eval)}/{len(spot_sample)}")
print(f"Silver label accuracy against that: {accuracy:.1%}")
print(f"Naive always-predict-majority-class baseline: {naive_baseline:.1%}")
print()
print("Confusion (silver_label vs true_relevant):")
print(pd.crosstab(spot_eval["silver_label"], spot_eval["true_relevant"]))
print()
print("Spot-checked queries that were part of the expanded gold standard's training set:")
print(spot_eval["query_id"].isin(labeled["query_id"]).value_counts())

Spot-checked with unanimous strong-judge agreement: 140/150
Silver label accuracy against that: 80.7%
Naive always-predict-majority-class baseline: 64.3%

Confusion (silver_label vs true_relevant):
true_relevant   0   1
silver_label         
0              39  16
1              11  74

Spot-checked queries that were part of the expanded gold standard's training set:
query_id
True    140
Name: count, dtype: int64
